# 环节 05 · 偏好对齐与 RLHF（配套 Notebook）

> 配套长文：[环节05-偏好对齐与RLHF详解.md](./环节05-偏好对齐与RLHF详解.md)
> 定位：BT 偏好模型 → 奖励模型 → KL 惩罚 → **闭式最优解**（通往 DPO 的那一步）。全部**纯 Python 标准库**。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 BT 概率 | §3 / §8.1 | Δ 越大越自信，梯度越接近 0（饱和） |
| §2 RM 损失 | §4 | 训 RM 就是最大化「chosen 比 rejected 好的概率」 |
| §3 KL 惩罚 | §6 | KL 逐 token 累加，不是整句一个数 |
| §4 闭式最优解 | §7 / §8.2 | `π* ∝ π_ref·exp(r/β)`，β 是"敢走多远"的旋钮 |
| §5 长度偏置 | §6 | RM 学到的伪特征怎么把策略带跑 |


## 1. Bradley-Terry：把"人更喜欢哪个"变成概率

```
P(y_w 优于 y_l | x) = σ( r(x, y_w) − r(x, y_l) ) = σ(Δ)
```

只看**奖励差 Δ**，不看绝对值 —— 这是后面 `Z(x)` 能被消掉的根源。


In [ ]:
import math

sig = lambda x: 1 / (1 + math.exp(-x))

print(f"{'Δ':>6} {'P(chosen 更好)':>15} {'loss = −log P':>15} {'|梯度| = σ(−Δ)':>16}")
for d in (0.0, 0.5, 1.0, 2.0, 4.0, 8.0):
    print(f"{d:>6.1f} {sig(d):>15.4f} {-math.log(sig(d)):>15.4f} {sig(-d):>16.4f}")
print("→ Δ 一大，P 趋近 1、loss 趋近 0、梯度趋近 0：BT 损失天然「不再管已经分对的样本」。")


## 2. 奖励模型：把偏好对当二分类来训

```
L_RM = − E[ log σ( r(x, y_w) − r(x, y_l) ) ]
```

- 只学到**序**（ranking），学不到"绝对分数"——奖励的零点/尺度是自由的（这正好是 DPO 能重参数化的原因）；
- 训练完就是一个**新模型**：要多占一份显存（RLHF 昂贵的第一来源）。


In [ ]:
# 一个 batch 的两对偏好，看 RM 损失怎么算
pairs = [("chosen_A", "rejected_A", 1.2, -0.3),      # (名, 名, r_chosen, r_rejected)
         ("chosen_B", "rejected_B", 0.1, 0.4)]       # 第二对是"学反了"的坏例

total = 0.0
for name_w, name_l, rw, rl in pairs:
    d = rw - rl
    loss = -math.log(sig(d))
    total += loss
    print(f"{name_w:>10} vs {name_l:<10} Δ = {d:+.2f}  loss = {loss:.4f}")
print(f"batch 平均 loss = {total / len(pairs):.4f}（第二对 Δ<0，被重罚）")


## 3. KL 惩罚的逐 token 实现

`per-token KL` 不是"整句一个数"，而是**每个 token 的 log 概率比之和**：

```
r_t = log π_θ(y_t) − log π_ref(y_t)
KL 项 = Σ_t r_t
```

LLM 的 KL 近似用逐 token 求和（或 k3 估计），所以**回答越长，同样的"偏离"被罚得越多**。


In [ ]:
logp_new = [-0.5, -1.2, -0.3, -2.0]                # 当前策略的逐 token log 概率
logp_ref = [-0.6, -1.0, -0.5, -1.8]                # 参考策略
ratio = [a - b for a, b in zip(logp_new, logp_ref)]
beta_kl = 0.05

print("逐 token log 比值 =", [round(x, 3) for x in ratio])
print("样本层 KL 惩罚（求和） =", round(sum(ratio), 4))
print("若用长度归一化（求均值） =", round(sum(ratio) / len(ratio), 4))
print(f"β = {beta_kl} 时进入奖励的惩罚项 = {-beta_kl * sum(ratio):+.4f}")
print("→ 求和 vs 求均值的差别，直接决定「长回答是否被系统性多罚」。")


## 4. 闭式最优解：`π* ∝ π_ref · exp(r/β)`

带 KL 正则的目标 `max_π E[r] − β·KL(π‖π_ref)` 可以**解析求解**：

```
π*(y|x) = (1/Z(x)) · π_ref(y|x) · exp( r(x,y) / β )
```

- **β 小** → 几乎只追最高奖励（语言质量不管，奖励黑客的模样）；
- **β 大** → `π* → π_ref`，策略不动、学不到东西。


In [ ]:
pi_ref, r = [0.5, 0.3, 0.2], [0.0, 1.0, 2.0]     # 3 选 1 玩具分布
print(f"{'β':>5}   π*（三选项概率）")
for beta in (0.1, 0.5, 2.0):
    w = [p * math.exp(ri / beta) for p, ri in zip(pi_ref, r)]
    Z = sum(w)
    pi = [x / Z for x in w]
    print(f"{beta:>5}   [{pi[0]:.4f} {pi[1]:.4f} {pi[2]:.4f}]")
print("  参考 π_ref = [0.5000 0.3000 0.2000]（β→∞ 的极限）")
print("→ β=0.1 时 99.99% 的概率压在最优选项上：这就是「奖励黑客」的数学长相。")


In [ ]:
# β → 0 / β → ∞ 两个极限，用更极端的 β 看趋势
for beta in (0.01, 0.05):
    w = [p * math.exp(ri / beta) for p, ri in zip(pi_ref, r)]
    Z = sum(w)
    pi = [x / Z for x in w]
    print(f"β={beta:<5} π* = [{pi[0]:.6f} {pi[1]:.6f} {pi[2]:.6f}]  ← 几乎退化成 argmax")


## 5. 长度偏置：RM 学到伪特征，策略就被带跑

如果训 RM 的数据里「回答更长 → 人更喜欢」系统性成立，RM 会把**长度**当成质量特征。RL 阶段策略立刻学会"越长越好"——真实质量没变，只是变长了。


In [ ]:
# 假设 RM 给"长度"额外加分 0.01/token，用一个简化奖励看策略会往哪走
print("若 RM 含长度伪特征（r 里含 +0.01·len）：")
for L in (40, 80, 160, 320):
    base, bonus = 1.0, 0.01 * L
    print(f"  长度 {L:>4}：真实质量分 {base:.1f}，RM 给分 {base + bonus:.2f}（虚高 {bonus:.1f}）")
print(f"→ 策略会把长度从 40 一路拉长到 320 甚至更长：长度 ×{320/40:.0f}，质量没动。")
print("  修法：成对长度匹配 + 长度归一化（SimPO 就自带这一项）。")


## 6. 小结与下钻

- **RLHF = RM + PPO + KL 正则**，BM 的偏好概率只看奖励差。
- **BT 损失自带"已分对就降权"**：`σ(−Δ)`，这一项在 DPO 梯度里原样出现。
- **闭式最优解 `π* ∝ π_ref·exp(r/β)` 是全部 DPO 推导的起点**——记住它，下一站就顺了。
- **β 控制"探索多远 vs 别忘本"**，量级随方法不同（RLHF 0.01~0.1 / DPO 0.1~0.5 / GRPO 0.001~0.04）。

下一站：[环节 06 · DPO 家族](./环节06-DPO家族与离线偏好优化详解.md)（把闭式解代回 BT，奖励模型就消失了）。
